In [ ]:

import json
from ipywidgets import AppLayout, Output, VBox
from ipyleaflet import Map, DrawControl, basemaps
from shapely.geometry import shape
from rasterio.mask import mask
import rasterio
import numpy as np
import geopandas as gpd
from collections import Counter


: 

In [ ]:

# Path to your raster file
LANDCOVER_PATH = "data/ESA_WorldCover_10m_2020_Map.tif"  # Ensure this file exists

# Optional: Define landcover class names
LANDCOVER_CLASSES = {
    10: "Trees",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare / sparse vegetation",
    70: "Snow and Ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen"
}


In [ ]:

output = Output()

# Create map widget
m = Map(center=(0, 0), zoom=2, basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True, layout={'height': '50vh'})

# Draw control
draw_control = DrawControl(polyline={}, circlemarker={}, circle={}, marker={})
draw_control.polygon = {"shapeOptions": {"color": "#6bc2e5", "fillOpacity": 0.5}}
draw_control.rectangle = {"shapeOptions": {"color": "#fca45d", "fillOpacity": 0.5}}

# Add draw control to map
m.add_control(draw_control)


In [ ]:
# Replace the AppLayout cell with a full-screen map and output panel using VBox

from ipywidgets import VBox, Layout
from IPython.display import display

# Set the map height to 100vh for full screen
m.layout = Layout(height='100vh')

# Display the map and output panel vertically
display(VBox([m, output]))

In [ ]:
@draw_control.on_draw
def handle_draw(target, action, geo_json):
    from ipyleaflet import Popup
    geom = shape(geo_json['geometry'])
    output.clear_output()
    with output:
        try:
            gdf = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[geom])
            gdf = gdf.to_crs('EPSG:3857')  # Project to metric CRS
            projected_geom = gdf.geometry[0]

            with rasterio.open(LANDCOVER_PATH) as src:
                clipped, transform = mask(src, [json.loads(gdf.to_json())['features'][0]['geometry']], crop=True)
                data = clipped[0].flatten()
                data = data[data != src.nodata]

                if data.size == 0:
                    msg = "No terrain data available in selected area."
                    print(msg)
                    popup = Popup(
                        location=[geom.centroid.y, geom.centroid.x],
                        child=Output(),
                        close_button=True,
                        auto_close=True,
                        close_on_escape_key=True
                    )
                    popup.child.append_stdout(msg)
                    m.popups = []
                    m.add_layer(popup)
                    return

                total_pixels = data.size
                counts = Counter(data)
                percentages = {k: (v / total_pixels) * 100 for k, v in counts.items()}

                # Prepare popup content
                msg = "Terrain type distribution (%):\n"
                for code, percent in percentages.items():
                    terrain_name = LANDCOVER_CLASSES.get(code, f"Class {code}")
                    msg += f"{terrain_name}: {percent:.2f}%\n"

                # Create and show popup at the centroid of the drawn geometry
                popup = Popup(
                    location=[geom.centroid.y, geom.centroid.x],
                    child=Output(),
                    close_button=True,
                    auto_close=False,
                    close_on_escape_key=True
                )
                popup.child.append_stdout(msg)
                m.popups = []
                m.add_layer(popup)

        except Exception as e:
            print(f"Error: {e}")

In [ ]:

@draw_control.on_draw
def handle_draw(target, action, geo_json):
    geom = shape(geo_json['geometry'])
    output.clear_output()
    with output:
        try:
            gdf = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[geom])
            gdf = gdf.to_crs('EPSG:3857')  # Project to metric CRS
            projected_geom = gdf.geometry[0]

            with rasterio.open(LANDCOVER_PATH) as src:
                clipped, transform = mask(src, [json.loads(gdf.to_json())['features'][0]['geometry']], crop=True)
                data = clipped[0].flatten()
                data = data[data != src.nodata]

                if data.size == 0:
                    print("No terrain data available in selected area.")
                    return

                total_pixels = data.size
                counts = Counter(data)
                percentages = {k: (v / total_pixels) * 100 for k, v in counts.items()}

                print("Terrain type distribution (%):")
                for code, percent in percentages.items():
                    terrain_name = LANDCOVER_CLASSES.get(code, f"Class {code}")
                    print(f"{terrain_name}: {percent:.2f}%")

        except Exception as e:
            print(f"Error: {e}")


In [ ]:

# app = AppLayout(
#     left_sidebar=None,
#     center=m,
#     right_sidebar=VBox([output]),
#     pane_widths=[0, 4, 3]
# )

# app
